# 06 SPEC EVOLUTION

> 核心：规范不是一成不变的文档，而是随业务演进的 living document。SPEC-DRIVEN 不是"一次规范无限生成"，而是"规范与生成代码共同演进"。

## 6.1 规范与代码的同步演化

### 传统模式的困境

```
需求变化 → 改代码 → 改规范（如果还记得） → 规范又过时了
```

**问题**：规范和代码脱节，规范变成"历史遗迹"。

### Spec Evolution 的思路

```
需求变化 → 新规范版本 → AI 重新生成 → Diff 对比 → 人类确认
                   ↓
              规范版本化
              (可追溯、可回滚)
```

**核心机制**：每次变更都从规范开始，AI 基于新规范重新生成，变化一目了然。

In [ ]:
# 规范版本化管理模拟
class SpecVersion:
    def __init__(self, version, content, changelog):
        self.version = version
        self.content = content
        self.changelog = changelog
        self.created_at = "2025-01-01"

    def diff(self, other):
        old_lines = set(self.content.split("\n"))
        new_lines = set(other.content.split("\n"))
        added = new_lines - old_lines
        removed = old_lines - new_lines
        return {"added": added, "removed": removed}

# 模拟版本演化
spec_v1 = SpecVersion(
    "1.0.0",
    """
API 设计规范:
- GET /users - 获取用户列表
- POST /users - 创建用户
""",
    "初始版本"
)

spec_v2 = SpecVersion(
    "1.1.0",
    """
API 设计规范:
- GET /users - 获取用户列表
- POST /users - 创建用户
- DELETE /users/:id - 删除用户  # 新增
""",
    "新增 DELETE /users/:id 接口"
)

diff = spec_v1.diff(spec_v2)
print(f"规范从 {spec_v1.version} 演化到 {spec_v2.version}")
print(f"新增: {diff['added']}")
print(f"移除: {diff['removed']}")

## 6.2 规范化变更管理

### 变更的类型

| 类型 | 说明 | 规范影响 |
|------|------|----------|
| 增补型 | 新增功能，不影响现有接口 | 小（添加新规范章节） |
| 破坏型 | 改变现有接口契约 | 大（需要版本升级） |
| 澄清型 | 规范描述模糊，澄清边界条件 | 中（更新描述性文字） |

### 破坏性变更的处理

```python
# API 版本策略
/api/v1/users  # 旧版
/api/v2/users  # 新版（破坏性变更）

# 规范版本策略
SPEC_v1.md  # 旧规范
SPEC_v2.md  # 新规范
```

In [ ]:
# 破坏性变更的版本策略
def handlebreaking_change(old_spec, new_spec, api_version):
    if api_version.startswith("v1"):
        print(f"使用规范版本: {old_spec.version} (兼容旧版)")
        return old_spec
    elif api_version.startswith("v2"):
        print(f"使用规范版本: {new_spec.version} (新版破坏性变更)")
        return new_spec

print("API 版本管理策略:")
handlebreaking_change(spec_v1, spec_v2, "v1")
handlebreaking_change(spec_v1, spec_v2, "v2")

## 6.3 AI 辅助规范演进

### AI 在规范演进中的角色

1. **变更检测**：AI 分析代码差异，自动识别规范需要更新的部分
2. **规范更新**：AI 根据变更内容，自动更新规范文档
3. **影响分析**：AI 分析规范变更对现有代码的影响

### 自动规范更新流程

```
代码 Diff → AI 分析变更类型 → AI 更新规范 → 人类确认 → 规范版本+1
```

In [ ]:
# AI 分析代码变更并生成规范更新建议
def ai_analyze_change(diff_content):
    suggestions = []

    if "def delete_user" in diff_content or "DELETE" in diff_content:
        suggestions.append({
            "type": "new_endpoint",
            "spec_section": "API 契约",
            "suggestion": "新增 DELETE /users/:id 接口规范",
            "required_fields": ["id", "deleted_at"]
        })

    if "soft_delete" in diff_content.lower() or "is_deleted" in diff_content.lower():
        suggestions.append({
            "type": "new_field",
            "spec_section": "数据模型",
            "suggestion": "User 数据模型新增 deleted_at 字段",
            "field_type": "datetime | null"
        })

    return suggestions

code_diff = """
+def delete_user(user_id):
+    user.is_deleted = True
+    user.deleted_at = datetime.now()
+    db.save(user)
"""

suggestions = ai_analyze_change(code_diff)
print("AI 生成的规范更新建议:")
for s in suggestions:
    print(f"  - {s}")

## 6.4 规范演进的可追溯性

### 为什么可追溯性重要？

- **调试**：当代码行为异常时，能快速定位是规范问题还是实现问题
- **回滚**：当新规范引入问题时，能回滚到上一个稳定版本
- **沟通**：新成员能理解规范的演进历史
- **审计**：满足合规要求，记录每次变更的原因

### Git 作为规范版本化的基础设施

```bash
git log SPEC.md
# commit a1b2c3d - feat: 新增 DELETE /users/:id 接口规范
# commit d4e5f6g - fix: 澄清边界条件 EMAIL_EXISTS
```

In [ ]:
spec_history = [
    {"version": "1.0.0", "date": "2025-01-01", "change": "初始版本", "author": "Alice"},
    {"version": "1.1.0", "date": "2025-01-15", "change": "新增 DELETE 接口", "author": "Bob"},
    {"version": "1.2.0", "date": "2025-02-01", "change": "澄清边界条件", "author": "Alice"},
]

print("规范版本历史:")
for entry in spec_history:
    print(f"  {entry['version']} ({entry['date']}) - {entry['change']} by {entry['author']}")

print(f"\n当前版本: {spec_history[-1]['version']}")
print(f"版本数: {len(spec_history)}")

## 6.5 实验结论

### Spec Evolution 的核心价值
- **规范化变更**：每次变更都从规范开始，避免规范与代码脱节
- **版本化管理**：规范可追溯、可对比、可回滚
- **AI 辅助演进**：AI 自动分析变更，生成规范更新建议
- **破坏性变更隔离**：通过 API 版本策略隔离破坏性变更

### 关键实践
- 规范变更必须伴随代码变更（不允许规范"裸更"）
- 每次规范变更必须有变更日志
- 破坏性变更必须升级主版本号

## 6.6 从规范到执行的完整流程

### Spec-Driven 开发流程（完整版）

```
需求 → 规范编写 → AI 生成 → 代码审查 → 合并 → 测试
  ↑______________|                    |
  |                                    |
  └────────────── 规范更新 ←───────────┘
```

### 各阶段产出物

| 阶段 | 产出物 | 负责人 |
|------|--------|--------|
| 需求分析 | 需求文档 | 产品经理 |
| 规范编写 | SPEC.md | 技术负责人 |
| AI 生成 | 代码 | AI |
| 代码审查 | Review 意见 | 人类 |
| 测试 | 测试报告 | QA |

In [ ]:
print("=== Spec-Driven 开发流程 (完整版) ===")
print()
print("Phase 1: 需求 → 规范")
print("  输入: 用户需求文档")
print("  输出: SPEC.md (规范文档)")
print()
print("Phase 2: 规范 → AI 生成")
print("  输入: SPEC.md")
print("  输出: 生成的代码")
print()
print("Phase 3: AI 生成 → 代码审查")
print("  输入: 生成的代码 + SPEC.md")
print("  输出: Review 意见 (合规性检查)")
print()
print("Phase 4: 代码审查 → 合并")
print("  输入: 通过审查的代码")
print("  输出: 合并到主分支")
print()
print("Phase 5: 测试")
print("  输入: 合并后的代码")
print("  输出: 测试报告")
print()
print("Phase 6: 规范更新 (如有变更)")
print("  输入: 测试发现的问题或新需求")
print("  输出: SPEC.md 新版本 (回到 Phase 1)")